# Introduction to State-Space Models

## The Cartpole
Let's begin with a classic toy problem in reinforcement learning: the cartpole. It's essentially two point masses connected by a massless rod, with the bottom mass (the "cart") able to move horizontally and the top mass (the "pole") able to swing freely. The goal is to apply forces to the cart to keep the pole balanced upright.

![../img/cartpole_sim.gif](../img/cartpole_sim.gif)


### The Nonlinear ODE

Letting $x$ be the horizontal position of the cart and $\theta$ be the angle of the pole from vertical, we can derive the equations of motion using Newton's laws or Lagrangian mechanics. With $m_c$ the cart mass, $m_p$ the pendulum mass, $l$ the pole half-length, and $M = m_c + m_p$ the total mass, the resulting nonlinear second-order ODEs are:
$$
\ddot{\theta} = \frac{M g \sin\theta - m_p l\, \dot{\theta}^2 \sin\theta \cos\theta}
                     {l\left(M - m_p \cos^2\!\theta\right)}
$$

$$
\ddot{x} = \frac{m_p l\left(\dot{\theta}^2 \sin\theta - \ddot{\theta}\cos\theta\right)}{M}
$$


In [ ]:
%load_ext autoreload
%autoreload 2

<div style="background: #E6F1FB; border-left: 4px solid #185FA5; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 8px;">
    <span style="font-size: 18px;">💡</span>
    <strong style="color: #185FA5; font-size: 15px;">Pause and Ponder</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    To simulate the system, four variables need to be evolved. Thus, we will say that the cartpole has four state variables. What are the four states?
  </p>
</div>

## Re-writing in State-Space Form

In general, we would like to write out systems in a standard form, which we will call the **state-space form**.

Rewritten as a **first-order system** $\dot{\mathbf{x}} = f(\mathbf{x})$:

$$
\mathbf{x} \equiv \begin{bmatrix} x \\ \dot{x} \\ \theta \\ \dot{\theta} \end{bmatrix}
\qquad
\frac{d}{dt}\mathbf{x}
=
\begin{bmatrix}
  \dot{x} \\
  \dfrac{m_p l\left(\dot{\theta}^2 \sin\theta - \ddot{\theta}\cos\theta\right)}{M} \\[8pt]
  \dot{\theta} \\[4pt]
  \dfrac{M g \sin\theta - m_p l\, \dot{\theta}^2 \sin\theta \cos\theta}
        {l\left(M - m_p \cos^2\!\theta\right)}
\end{bmatrix}
$$
There are a couple of different ways we can express the fact that parameters are involved. Here, we will use notation that mirrors Python code. Letting $\mathbf{p} = (m_c, m_p, l, g)$ be the parameters of the system, we can write:
$$
\dot{\mathbf{x}} = f(\mathbf{x}; \mathbf{p})
$$

## Discretizing in Time
In practice, we will almost always discretize in time to perform simulation on computers. There are a lot of different numerical integration methods, but ultimately they all reuslt in the following form:
$$
\mathbf{x}_{t+1} = f(\mathbf{x}_t; \mathbf{p})
$$
Where $t$ is the index of the time step.

## Actions

So far we have studied the *free* cartpole — the pole simply falls over.
In practice we can push the cart: a horizontal force ${\color{teal} F}$ (N) applied to the cart changes the equations of motion.

We call ${\color{teal} F}$ an **action** and collect it in a vector $\mathbf{a}$:

$$
\mathbf{a} = \begin{bmatrix} {\color{teal} F} \end{bmatrix} \in \mathbb{R}^1
$$

The controlled dynamics become:

$$
\frac{d}{dt}\begin{bmatrix} x \\ \dot{x} \\ \theta \\ \dot{\theta} \end{bmatrix}
=
\begin{bmatrix}
  \dot{x} \\[4pt]
  \dfrac{{\color{teal} F} + m_p l\left(\dot{\theta}^2 \sin\theta - \ddot{\theta}\cos\theta\right)}{M} \\[10pt]
  \dot{\theta} \\[4pt]
  \dfrac{M g \sin\theta - \cos\theta\!\left({\color{teal} F} + m_p l\,\dot{\theta}^2 \sin\theta\right)}
        {l\left(M - m_p \cos^2\!\theta\right)}
\end{bmatrix}
$$

In general, dynamical models with actions are expressed in continuous time as:
$$
\dot{\mathbf{x}} = f(\mathbf{x}, \mathbf{a}; \mathbf{p})
$$
or in discrete time as:
$$
\mathbf{x}_{t+1} = f(\mathbf{x}_t, \mathbf{a}_t; \mathbf{p})
$$

<div style="background: #E6F1FB; border-left: 4px solid #185FA5; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 8px;">
    <span style="font-size: 18px;">💡</span>
    <strong style="color: #185FA5; font-size: 15px;">Pause and Ponder</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    Is it reasonable to say we can command Force?
  </p>
</div>

## Observations
Consider the real cartpole below. How can we know its system state? We would need sensors, which will inevitably be imperfect and have some amount of both noise and systematic biases involved. That is, the set of **observations**, $\mathbf{o}$, we can make about the system does not directly correspond to the system state, $\mathbf{x}$. That is $\mathbf{o} \neq \mathbf{x}$.

![../img/real_cartpole.png](../img/real_cartpole.png)

It is important to take the distinction between observations and state into account when modelling. This is done with an **observation function**, which maps state, actions, and model parameters to observations:

$$\mathbf{o} = O(\mathbf{x}, \mathbf{a}; \mathbf{p})$$

Since noise is so ubiquitous, it's more accurate to say that observations are sampled from a distribution:
$$\mathbf{o}\sim O(\mathbf{x}, \mathbf{a}; \mathbf{p})$$


For the purposes of this exercise, we will assume that we have sensors that can measure the cart position and pole angle with some amount of noise. The observations are thus a linear model:

$$
\mathbf{o} = C\mathbf{x} + \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},\, R)
$$

The **observation matrix** $C$ picks out $x$ and $\theta$ from the state vector $\mathbf{x} = [x,\, \dot{x},\, \theta,\, \dot{\theta}]^\top$:

$$
C = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \end{bmatrix}
$$

so that $C\mathbf{x} = [x,\, \theta]^\top$ exactly. The **noise covariance** $R$ is diagonal since the two sensor channels are independent:

$$
R = \begin{bmatrix} \sigma_x^2 & 0 \\ 0 & \sigma_\theta^2 \end{bmatrix}
$$


## State-Space Models and POMDPs
Putting everything together, we can define a general nonlinear state space model as a pair of functions:
$$
\begin{aligned}
\mathbf{x}_{t+1} &= f(\mathbf{x}_t, \mathbf{a}_t; \mathbf{p}) \\
\mathbf{o}_t &= O(\mathbf{x}_t, \mathbf{a}_t; \mathbf{p})
\end{aligned}
$$
in the deterministic case, or as a pair of distributions in the stochastic case:
$$
\begin{aligned}
\mathbf{x}_{t+1} &\sim f(\mathbf{x}_t, \mathbf{a}_t; \mathbf{p}) \\
\mathbf{o}_t &\sim O(\mathbf{x}_t, \mathbf{a}_t; \mathbf{p})
\end{aligned}
$$

This is an extremely fundamental formalism with some very important structure. It captures some highly important assumptions about the world:
 - There is some underlying set of variables governing the system evolution across time
 - Some of the variables are variables we can do something about (actions), others evolve as mother nature decides (states)
 - We can't observe all of the relevant underlying variables, but rather we observe a set of variables that are generated from the underlying variables in some way (observations)

### POMDPs and Reinforcement Learning
We have now stumbled most of our way to the formalism that defines most of what reinforcement learning (RL) aims to solve: the **partially observable Markov decision process** (POMDP). One can think of a POMDP as a state-space model with the addition of a reward function $r(\mathbf{x}, \mathbf{a})$ that provides a measure of goodness to each state/action (e.g. Q being big is good, plasma melting the wall is bad). The goal of RL is to find a policy function that maps observations to actions in a way that maximizes the expected reward over time. We define it as such:
$$
\begin{aligned}
\max_\pi \quad &\mathbb{E}\!\left[\sum_{t=0}^\infty  r(\mathbf{x}_t, \mathbf{a}_t)\right] \\
& \mathbf{x}_{t+1} \sim f(\mathbf{x}_t, \mathbf{a}_t; \mathbf{p})\\
&\mathbf{o}_t \sim O(\mathbf{x}_t, \mathbf{a}_t; \mathbf{p})\\
& \mathbf{a}_t = \pi(\mathbf{o}_t)
\end{aligned}
$$

### On Terminology
There is a whole zoo of different names for models that are equivalent or analogous to state-space models. For example, the RL world usually calls the "transition and observation functions" or the "world model" (although the term world model seems to be shifting in meaning to some extent lately).

## Running a Simulation

<div style="background: #EDFAF3; border-left: 4px solid #1A7A4A; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 10px;">
    <span style="font-size: 18px;">✏️</span>
    <strong style="color: #1A7A4A; font-size: 15px;">Exercise — Feedforward Control</strong>
  </div>
    <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
        Run a simulation of the cartpole. Then try introducing a constant force to see what happens! When actions are pre-programmed functions of time, we call this feedforward control. What might 'feedback control' be?
    </p>
</div>

In [ ]:
import sys

sys.path.insert(0, "../src")

import jax
import jax.numpy as jnp
from IPython.display import HTML

from state_space_model.systems.cartpole import CartPole, CartPoleParams, animate_trajectory

jax.config.update("jax_enable_x64", True)

# --- Simulation setup ---
params = CartPoleParams()
system = CartPole()

dt = 0.02  # timestep (s)
duration = 3.0  # total simulation time (s)
T = int(duration / dt)

# Initial state: cart at rest, pole tilted 10° from vertical
theta0 = jnp.deg2rad(10.0)
x0 = jnp.array([0.0, 0.0, theta0, 0.0])  # [x, x_dot, theta, theta_dot]

# Zero force throughout (free fall)
actions = jnp.zeros((T, 1))  # TODO: change me to non-zero forces to see what happens!

In [ ]:
from state_space_model.systems.cartpole import ObservationNoiseParams

key = jax.random.PRNGKey(0)
noise = ObservationNoiseParams(x_std=0.005, theta_std=0.005)

# --- Run the rollout ---
trajectory, observations = system.rollout_with_obs(
    x0, actions, params, key=key, noise=noise, dt=dt
)

# --- Visualize ---
anim = animate_trajectory(trajectory, actions, params, dt=dt, observations=observations)
HTML(anim.to_jshtml())

## Real2Sim
<div style="background: #E6F1FB; border-left: 4px solid #185FA5; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 8px;">
    <span style="font-size: 18px;">💡</span>
    <strong style="color: #185FA5; font-size: 15px;">Pause and Ponder</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    Consider the real cartpole below. We want to use our simulator to predict what it will do. How might we go about doing this? What challenges might we encounter?
  </p>
</div>


![../img/real_cartpole.png](../img/real_cartpole.png)


The problem of determining the state of a real system $\mathbf{x}$ from sensor observations $\mathbf{o}$ is often called **state estimation** and is an entire field of study in its own right.

## ✏️ Exercise — Sensitivity to Initial Conditions

To run the simulation, we need to determine what the true initial state of the cartpole is. Unfortunately, our sensors are imperfect, so we have both noise and systematic errors. Let's run two simulations to determine what happens if our measurements of the initial state are slightly off.


**Your task:** Adjust the values of `theta_error` and `theta_dot_error` to non-zero values and observe how the trajectories diverge over time. How much error can we tolerate before our predictions become highly inaccurate?


<div style="background: #E6F1FB; border-left: 4px solid #185FA5; border-radius: 6px; padding: 16px 20px; font-family: sans-serif;">
  <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 8px;">
    <span style="font-size: 18px;">💡</span>
    <strong style="color: #185FA5; font-size: 15px;">Pause and Ponder</strong>
  </div>
  <p style="color: #0C447C; font-size: 14px; margin: 0; line-height: 1.6;">
    Given what we have learned from this exercise, how should we go about getting useful predictions of the real cartpole's behavior?
  </p>
</div>

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import jax.numpy as jnp

# ── "True" initial state ────────────────────────────────────────────────────────
theta_true = jnp.deg2rad(1.0)
x0_true = jnp.array([0.0, 0.0, theta_true, 0.0])  # [x, x_dot, theta, theta_dot]

# ── TODO: CHANGE ME ─────────────
theta_error = jnp.deg2rad(
    0.0
)  # TODO: change me to non-zero angle error to see what happens!
theta_dot_error = jnp.deg2rad(
    0.0
)  # TODO: change me to non-zero angular velocity error to see what happens!

x0_measured = x0_true + jnp.array([0.0, 0.0, theta_error, theta_dot_error])

# ── Run both rollouts ─────────────────────────────────────────────────────────
trajectory_true = system.rollout(x0_true, actions, params, dt=dt)
trajectory_measured = system.rollout(x0_measured, actions, params, dt=dt)

time = jnp.linspace(0, duration, T + 1)

# ── Plot ──────────────────────────────────────────────────────────────────────
state_labels = [
    (r"$x$ (m)", "Cart position"),
    (r"$\dot{x}$ (m/s)", "Cart velocity"),
    (r"$\theta$ (rad)", "Pole angle"),
    (r"$\dot{\theta}$ (rad/s)", "Pole angular velocity"),
]

fig = plt.figure(figsize=(11, 7))
gs = gridspec.GridSpec(
    3,
    2,
    figure=fig,
    height_ratios=[1, 1, 0.7],
    hspace=0.6,
    wspace=0.38,
    left=0.08,
    right=0.97,
    top=0.93,
    bottom=0.09,
)

# State comparison panels
for i, (ylabel, title) in enumerate(state_labels):
    ax = fig.add_subplot(gs[i // 2, i % 2])
    ax.plot(time, trajectory_true[:, i], color="#185FA5", linewidth=1.5, label="true")
    ax.plot(
        time,
        trajectory_measured[:, i],
        color="#E05A2B",
        linewidth=1.5,
        label="perturbed $x_0$",
        linestyle="--",
    )
    ax.set_title(title, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.grid(True, alpha=0.3)
    if i >= 2:
        ax.set_xlabel("Time (s)", fontsize=8)
    if i == 0:
        ax.legend(fontsize=8, loc="upper left")

# State error norm: ||x_true(t) - x_sim(t)||
ax_err = fig.add_subplot(gs[2, :])
error_norm = jnp.linalg.norm(trajectory_true - trajectory_measured, axis=1)
ax_err.plot(time, error_norm, color="#1A7A4A", linewidth=1.5)
ax_err.fill_between(time, error_norm, alpha=0.15, color="#1A7A4A")
ax_err.set_title(
    "State error  " + r"$\|\mathbf{x}_\mathrm{true}(t) - \mathbf{x}_\mathrm{sim}(t)\|$",
    fontsize=9,
)
ax_err.set_ylabel("Error (mixed units)", fontsize=8)
ax_err.set_xlabel("Time (s)", fontsize=8)
ax_err.grid(True, alpha=0.3)

plt.show()

## Implications for Tokamak Simulation
So we have seen that if the dynamics of a system has chaos/sensitivity involved, even if the physics is completely well understood, we still have challenges in making accurate predictions because we don't know what the initial state of the system is.

One of the challenges in tokamak simulation is that higher fidelity models typically require **more** system state (e.g. POPCONs just require some 0D variables, while kinetic treatments require full distribution functions). If the state is difficult to observe and small errors in the state can lead to large errors in the predictions, the higher fidelity model may not actually lead to better predictions of the real system!

So in summary, there are two key implications:
1. Higher fidelity => more state initialization requirements => may not lead to better predictions
2. Sensitivity to initial conditions requires the use of parallel simulations

These challenges leads the author to make the following anecdotal observation:

![../img/nonmonotonic_fidelity.png](../img/nonmonotonic_fidelity.png)
